# Fisher-KPP RK4 Demo

This notebook runs 1D and 2D Fisher-KPP RK4 examples in exact traveling-wave regimes. The RK4 method is unchanged; the benchmark problem is specified so numerical fields can be compared directly with closed-form solutions.

- 1D PDE: `u_t = u_xx + u(1-u)`, `D=1`, `r=1`.
- 1D exact speed: `c = 5 / sqrt(6)`.
- 1D exact wave: `u(x,t) = (1 + exp((x - c t - x0) / sqrt(6)))^-2`.
- 1D grid: `x in [-20, 20]`, `T=10`, `Nx=201`, `dt=0.005`.
- 2D PDE: `u_t = u_xx + u_yy + u(1-u)`.
- 2D exact wave: `[0.5 tanh((x+y)/(4 sqrt(3)) + 5t/12) + 0.5]^2`.
- 2D grid: `x,y in [-15, 15]`, `T=3`, `61x61`, `dt=0.01`, exact Dirichlet boundary values.


In [ ]:
%matplotlib inline

from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

REPO_URL = "https://github.com/rladbsco24/fisher-pinn.git"
REPO_BRANCH = "main"


def _has_rk4_project(root: Path) -> bool:
    return (root / "fisher-kpp-rk4" / "src" / "fisher_kpp_rk4").exists()


def _prepare_colab_repo(repo_dir: Path) -> Path:
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)], check=True)
    elif (repo_dir / ".git").exists():
        subprocess.run(["git", "fetch", "--depth", "1", "origin", REPO_BRANCH], cwd=repo_dir, check=True)
        subprocess.run(["git", "checkout", "--force", "FETCH_HEAD"], cwd=repo_dir, check=True)
    return repo_dir.resolve()


PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if _has_rk4_project(candidate):
        PROJECT_ROOT = candidate
        break

if not _has_rk4_project(PROJECT_ROOT) and Path("/content").exists():
    PROJECT_ROOT = _prepare_colab_repo(Path("/content/fisher-pinn"))

if not _has_rk4_project(PROJECT_ROOT):
    raise RuntimeError("Run this notebook inside the fisher-pinn repository, or use Colab with network access enabled.")

RK4_ROOT = PROJECT_ROOT / "fisher-kpp-rk4"
SRC_DIR = RK4_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fisher_kpp_rk4 import check_rk4_stability, estimate_front_speed, solve_rk4, solve_rk4_2d
from fisher_kpp_rk4.config import (
    BOX_2D,
    D,
    D_2D,
    GRID_2D,
    L,
    Nt,
    Nt_2d,
    Nx,
    T,
    T_2D,
    ablowitz_zeppetella_exact,
    generalized_fisher_kpp_exact_2d,
    c,
    dt,
    dt_2d,
    dx,
    dx_2d,
    initial_condition,
    initial_condition_2d,
    left_bc,
    r,
    r_2D,
    right_bc,
    x,
    x_2d,
    y_2d,
)

OUTPUT_DIR = RK4_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"project root: {PROJECT_ROOT}")
print(f"rk4 root: {RK4_ROOT}")
print(f"src dir: {SRC_DIR}")


In [ ]:
info_1d = check_rk4_stability(dx=dx, dt=dt, D=D, r=r, dim=1)
print(f"1D: D={D}, r={r}, L={L}, T={T}")
print(f"Nx={Nx}, dx={dx:.6g}, Nt={Nt}, dt={dt:.6g}")
print(f"Ablowitz-Zeppetella speed c = {c:.6g}")
print(f"Practical dt safe? {info_1d['is_practically_safe']} (limit={info_1d['dt_practical']:.6g})")

result_1d = solve_rk4(
    x=x,
    dt=dt,
    Nt=Nt,
    D=D,
    r=r,
    initial_condition=initial_condition,
    left_bc=left_bc,
    right_bc=right_bc,
    save_interval=1.0,
    exact_solution=ablowitz_zeppetella_exact,
)

c_num = estimate_front_speed(result_1d["times"], result_1d["fronts"], t_min=1.0, x_max=x[-1])
print(f"Estimated front speed = {c_num:.6g}")
print(f"Final relative L2 vs exact = {float(result_1d['relative_l2_final']):.3e}")


In [ ]:
plt.figure(figsize=(8, 5))
for t, u, u_ex in zip(result_1d["times"], result_1d["snapshots"], result_1d["exact_snapshots"]):
    plt.plot(result_1d["x"], u, label=f"RK4 t={t:.0f}")
    plt.plot(result_1d["x"], u_ex, "--", linewidth=1.0, color="black", alpha=0.35)
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.ylim(-0.05, 1.05)
plt.title("1D Ablowitz-Zeppetella wave: RK4 and exact")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(result_1d["times"], result_1d["fronts"], marker="o", label="RK4")
exact_front_05 = c * result_1d["times"] + np.sqrt(6.0) * np.log(np.sqrt(2.0) - 1.0)
axes[0].plot(result_1d["times"], exact_front_05, "--", label="exact u=0.5 front")
axes[0].set_xlabel("t")
axes[0].set_ylabel("front position, u=0.5")
axes[0].set_title("1D front propagation")
axes[0].legend()
axes[1].plot(result_1d["x"], result_1d["abs_error_final"])
axes[1].set_xlabel("x")
axes[1].set_ylabel("absolute error")
axes[1].set_title("Final-time absolute error")
plt.show()


In [ ]:
info_2d = check_rk4_stability(dx=dx_2d, dt=dt_2d, D=D_2D, r=r_2D, dim=2)
print(f"2D: D={D_2D}, r={r_2D}, box={BOX_2D}, T={T_2D}")
print(f"grid={GRID_2D}x{GRID_2D}, dx={dx_2d:.6g}, Nt={Nt_2d}, dt={dt_2d:.6g}")
print(f"Practical dt safe? {info_2d['is_practically_safe']} (limit={info_2d['dt_practical']:.6g})")

result_2d = solve_rk4_2d(
    x=x_2d,
    y=y_2d,
    dt=dt_2d,
    Nt=Nt_2d,
    D=D_2D,
    r=r_2D,
    initial_condition=initial_condition_2d,
    save_interval=0.05,
    boundary_condition="dirichlet_exact",
    exact_solution=generalized_fisher_kpp_exact_2d,
)
print(f"Final mean mass = {result_2d['mass'][-1]:.6g}")
print(f"Final area u>=0.05 = {result_2d['area_ge_0.05'][-1]:.6g}")
print(f"Final area u>=0.10 = {result_2d['area_ge_0.10'][-1]:.6g}")
print(f"Final relative L2 vs exact = {float(result_2d['relative_l2_final']):.3e}")


In [ ]:
snapshots = result_2d["snapshots"]
times = result_2d["times"]
panel_idx = np.unique(np.linspace(0, len(times) - 1, 4, dtype=int))
fig, axes = plt.subplots(1, len(panel_idx), figsize=(4.0 * len(panel_idx), 3.4), constrained_layout=True)
if len(panel_idx) == 1:
    axes = [axes]
for ax, idx in zip(axes, panel_idx):
    im = ax.imshow(
        snapshots[idx].T,
        origin="lower",
        extent=[result_2d["x"][0], result_2d["x"][-1], result_2d["y"][0], result_2d["y"][-1]],
        vmin=0.0,
        vmax=1.0,
        cmap="cividis",
    )
    ax.set_title(f"t={times[idx]:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
fig.colorbar(im, ax=axes, shrink=0.82)
fig.suptitle("2D generalized Fisher-KPP exact-wave RK4 snapshots")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8), constrained_layout=True)
for ax, field, title, vmax in [
    (axes[0], result_2d["exact_final"], "exact final", 1.0),
    (axes[1], result_2d["abs_error_final"], "absolute error", None),
]:
    im = ax.imshow(
        field.T,
        origin="lower",
        extent=[result_2d["x"][0], result_2d["x"][-1], result_2d["y"][0], result_2d["y"][-1]],
        vmin=0.0,
        vmax=vmax,
        cmap="cividis" if title.startswith("exact") else "inferno",
    )
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    fig.colorbar(im, ax=ax, shrink=0.82)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(result_2d["times"], result_2d["mass"], label="mean mass")
plt.plot(result_2d["times"], result_2d["area_ge_0.05"], label="area u>=0.05")
plt.plot(result_2d["times"], result_2d["area_ge_0.10"], label="area u>=0.10")
plt.xlabel("t")
plt.ylabel("fraction")
plt.title("2D mass and front-area diagnostics")
plt.legend()
plt.tight_layout()
plt.show()


## RK4 Grid Comparison Tables

This section generates report-style RK4 tables for 1D/2D spatial-grid and time-step comparisons. The columns follow the numerical-method comparison format, but use `rk4_stages_per_step` and `stability_safe` because RK4 has no Newton iteration.


In [ ]:
from IPython.display import Image, display

subprocess.run([sys.executable, str(RK4_ROOT / "scripts" / "run_convergence.py")], cwd=PROJECT_ROOT, check=True)
TABLE_DIR = RK4_ROOT / "outputs" / "tables"
for table_name in [
    "rk4_1d_spatial_comparison.png",
    "rk4_1d_time_comparison.png",
    "rk4_1d_temporal_reference_convergence.png",
    "rk4_2d_spatial_comparison.png",
    "rk4_2d_time_comparison.png",
    "rk4_2d_temporal_reference_convergence.png",
]:
    display(Image(filename=str(TABLE_DIR / table_name)))


## Report-Style 2D RK4 3D Figures

The following cells reproduce the required generalized Fisher-KPP visual diagnostics: centerline traces, 3D solution surfaces, and 3D absolute-error surfaces at `t = 0, 2, 4, 6, 8`. The solver, grid, time step, exact initial condition, and exact Dirichlet boundaries are the same as the 2D RK4 benchmark; only the visualization horizon is extended to `T = 8`.


In [ ]:
REPORT_2D_TIMES = (0.0, 2.0, 4.0, 6.0, 8.0)
report_t_final = max(REPORT_2D_TIMES)
report_nt = int(round(report_t_final / dt_2d))
result_2d_report = solve_rk4_2d(
    x=x_2d,
    y=y_2d,
    dt=dt_2d,
    Nt=report_nt,
    D=D_2D,
    r=r_2D,
    initial_condition=initial_condition_2d,
    save_interval=2.0,
    boundary_condition="dirichlet_exact",
    exact_solution=generalized_fisher_kpp_exact_2d,
)
np.savez(
    OUTPUT_DIR / "fisher_kpp_rk4_2d_report_visualization.npz",
    **result_2d_report,
    D=D_2D,
    r=r_2D,
    box=BOX_2D,
    T=report_t_final,
    dx=dx_2d,
    dt=dt_2d,
)
print(f"Saved report visualization NPZ to {OUTPUT_DIR / 'fisher_kpp_rk4_2d_report_visualization.npz'}")


In [ ]:
from matplotlib import colors

xx_report, yy_report = np.meshgrid(result_2d_report["x"], result_2d_report["y"], indexing="ij")
times_report = np.asarray(result_2d_report["times"], dtype=np.float64)
snapshots_report = np.asarray(result_2d_report["snapshots"], dtype=np.float64)
y_mid_idx = len(result_2d_report["y"]) // 2

def nearest_snapshot_index(times, target):
    return int(np.argmin(np.abs(np.asarray(times, dtype=np.float64) - float(target))))

def save_3d_surface(path, zz, *, title, zlabel, vmin, vmax, cmap="viridis"):
    fig = plt.figure(figsize=(6.8, 4.8), constrained_layout=True)
    ax = fig.add_subplot(111, projection="3d")
    norm = colors.Normalize(vmin=vmin, vmax=vmax)
    surf = ax.plot_surface(
        xx_report,
        yy_report,
        zz,
        cmap=cmap,
        norm=norm,
        linewidth=0.0,
        antialiased=True,
        rstride=1,
        cstride=1,
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel(zlabel, labelpad=5)
    ax.view_init(elev=28, azim=-58)
    ax.set_xlim(float(xx_report.min()), float(xx_report.max()))
    ax.set_ylim(float(yy_report.min()), float(yy_report.max()))
    ax.set_zlim(vmin, vmax)
    cbar = fig.colorbar(surf, ax=ax, shrink=0.72, pad=0.08)
    cbar.set_label(zlabel, fontsize=8)
    fig.savefig(path, dpi=180)
    plt.show()

plt.figure(figsize=(7.0, 4.2))
for target in REPORT_2D_TIMES:
    idx = nearest_snapshot_index(times_report, target)
    plt.plot(result_2d_report["x"], snapshots_report[idx, :, y_mid_idx], label=f"t = {times_report[idx]:.0f}")
plt.xlabel("x")
plt.ylabel("u(x,0,t)")
plt.ylim(-0.03, 1.03)
plt.grid(alpha=0.35)
plt.legend(fontsize=8)
plt.title("2D generalized Fisher-KPP centerline over time")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "centerline_2d_exact_wave.png", dpi=200)
plt.show()

for target in REPORT_2D_TIMES:
    idx = nearest_snapshot_index(times_report, target)
    t_value = float(times_report[idx])
    numerical = snapshots_report[idx]
    exact = np.asarray(generalized_fisher_kpp_exact_2d(xx_report, yy_report, t_value), dtype=np.float64)
    abs_error = np.abs(numerical - exact)
    tag = f"t{int(round(t_value)):02d}"
    save_3d_surface(
        OUTPUT_DIR / f"surface_2d_exact_wave_{tag}.png",
        numerical,
        title=f"2D generalized Fisher-KPP 3D Surface at t = {t_value:.0f}",
        zlabel="u(x,y,t)",
        vmin=0.0,
        vmax=1.0,
    )
    save_3d_surface(
        OUTPUT_DIR / f"absolute_error_2d_surface_{tag}.png",
        abs_error,
        title=f"Absolute Error 3D Surface at t = {t_value:.0f}",
        zlabel="absolute error",
        vmin=0.0,
        vmax=max(float(np.nanmax(abs_error)), 1.0e-12),
    )

print(f"Saved report-style 2D figures under {OUTPUT_DIR}")


In [ ]:
np.savez(OUTPUT_DIR / "fisher_kpp_rk4_1d_results.npz", **result_1d, D=D, r=r, L=L, T=T, dx=dx, dt=dt)
np.savez(OUTPUT_DIR / "fisher_kpp_rk4_2d_results.npz", **result_2d, D=D_2D, r=r_2D, box=BOX_2D, T=T_2D, dx=dx_2d, dt=dt_2d)
print("Saved 1D and 2D NPZ outputs under outputs/.")


## Matched Visualization Format

The following cell reproduces the same visualization layout used in `KPP_Fisher_trapezoidal_AZ_exact`: 1D profiles, 1D absolute error, 2D centerline, 2D surfaces, heatmaps, and numerical/exact/error 3D surfaces.


In [ ]:

# Matched visualization format against KPP_Fisher_trapezoidal_AZ_exact
# This cell solves the same 1D Ablowitz-Zeppetella and 2D generalized Fisher-KPP exact benchmarks with RK4.
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

RK4_ROOT = Path.cwd()
if (RK4_ROOT / "fisher-kpp-rk4").exists():
    RK4_ROOT = RK4_ROOT / "fisher-kpp-rk4"
elif not (RK4_ROOT / "src" / "fisher_kpp_rk4").exists():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "fisher-kpp-rk4"
        if (candidate / "src" / "fisher_kpp_rk4").exists():
            RK4_ROOT = candidate
            break
SRC_DIR = RK4_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fisher_kpp_rk4 import solve_rk4, solve_rk4_2d
from fisher_kpp_rk4.config import (
    D, r, T, dt, Nt, x,
    D_2D, r_2D, T_2D,
    ablowitz_zeppetella_exact,
    generalized_fisher_kpp_exact_2d,
    initial_condition,
    initial_condition_2d,
    left_bc,
    right_bc,
    x_left_2d,
    x_right_2d,
    y_bottom_2d,
    y_top_2d,
)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 180})
OUTPUT_DIR = RK4_ROOT / "outputs" / "matched_visualizations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1D profile and absolute-error plots
rk4_1d = solve_rk4(
    x=x,
    dt=dt,
    Nt=Nt,
    D=D,
    r=r,
    initial_condition=initial_condition,
    left_bc=left_bc,
    right_bc=right_bc,
    save_interval=2.0,
    exact_solution=ablowitz_zeppetella_exact,
)
plt.figure(figsize=(8, 5))
for t, u in zip(rk4_1d["times"], rk4_1d["snapshots"]):
    plt.plot(rk4_1d["x"], u, label=f"t = {float(t):g}")
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.title("1D KPP-Fisher equation using RK4 method")
plt.grid(True, alpha=0.35)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rk4_1d_profiles.png")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(rk4_1d["x"], np.abs(rk4_1d["u_final"] - rk4_1d["exact_final"]), color="tab:red")
plt.xlabel("x")
plt.ylabel("absolute error")
plt.title(f"1D absolute error at t = {float(T):g}")
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rk4_1d_absolute_error.png")
plt.show()

# 2D centerline, surface, heatmap, and numerical/exact/error surface plots
x2 = np.linspace(x_left_2d, x_right_2d, 61)
y2 = np.linspace(y_bottom_2d, y_top_2d, 61)
X2, Y2 = np.meshgrid(x2, y2, indexing="ij")
rk4_2d_vis = solve_rk4_2d(
    x=x2,
    y=y2,
    dt=0.01,
    Nt=800,
    D=D_2D,
    r=r_2D,
    initial_condition=initial_condition_2d,
    save_interval=2.0,
    boundary_condition="dirichlet_exact",
    exact_solution=generalized_fisher_kpp_exact_2d,
)
center_j = len(y2) // 2
plt.figure(figsize=(8, 5))
for t, U in zip(rk4_2d_vis["times"], rk4_2d_vis["snapshots"]):
    plt.plot(x2, U[:, center_j], label=f"t = {float(t):g}")
plt.xlabel("x")
plt.ylabel("u(x,0,t)")
plt.title("2D generalized Fisher-KPP centerline over time")
plt.grid(True, alpha=0.35)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "rk4_2d_centerline.png")
plt.show()

for t, U in zip(rk4_2d_vis["times"], rk4_2d_vis["snapshots"]):
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    surf = ax.plot_surface(X2, Y2, U, cmap="viridis", edgecolor="none", alpha=0.96)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("u(x,y,t)")
    ax.set_title(f"2D generalized Fisher-KPP 3D Surface at t = {float(t):g}")
    fig.colorbar(surf, ax=ax, shrink=0.65, label="u(x,y,t)")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"rk4_2d_surface_t{str(float(t)).replace('.', 'p')}.png")
    plt.show()

vmin = float(np.min(rk4_2d_vis["snapshots"]))
vmax = float(np.max(rk4_2d_vis["snapshots"]))
fig, axes = plt.subplots(1, len(rk4_2d_vis["times"]), figsize=(4 * len(rk4_2d_vis["times"]), 4), constrained_layout=True)
for ax, t, U in zip(np.ravel(axes), rk4_2d_vis["times"], rk4_2d_vis["snapshots"]):
    im = ax.imshow(U.T, origin="lower", extent=[x2.min(), x2.max(), y2.min(), y2.max()], vmin=vmin, vmax=vmax, cmap="viridis", aspect="equal")
    ax.set_title(f"t={float(t):g}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
fig.colorbar(im, ax=np.ravel(axes).tolist(), shrink=0.75, label="u(x,y,t)")
fig.suptitle("2D generalized Fisher-KPP heatmap snapshots", y=1.02)
fig.savefig(OUTPUT_DIR / "rk4_2d_heatmap_snapshots.png")
plt.show()

for t, U_num in zip(rk4_2d_vis["times"], rk4_2d_vis["snapshots"]):
    U_exact = generalized_fisher_kpp_exact_2d(X2, Y2, float(t))
    Abs_error = np.abs(U_num - U_exact)
    for label, data, cmap, zlabel in [
        ("Numerical", U_num, "viridis", "u_num"),
        ("Exact", U_exact, "viridis", "u_exact"),
        ("Absolute Error", Abs_error, "magma", "|u_num-u_exact|"),
    ]:
        fig = plt.figure(figsize=(9, 6))
        ax = fig.add_subplot(111, projection="3d")
        surf = ax.plot_surface(X2, Y2, data, cmap=cmap, edgecolor="none")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel(zlabel)
        ax.set_title(f"{label} 3D Surface at t = {float(t):g}")
        fig.colorbar(surf, ax=ax, shrink=0.65, label=zlabel)
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / f"rk4_2d_{label.lower().replace(' ', '_')}_t{str(float(t)).replace('.', 'p')}.png")
        plt.show()
